# Study ordering of io users for different approaches
## # Co-retweets vs # Days users synchronize (rt in less than 60 sec.)
### Code is contained in `synchronous_repeated_detection.py`

In [12]:
from synchronous_repeated_detection import import_data, count_coretweets, filter_RTs, coincide_ignoring_tweet, coincide_shared_tweets, coincide_same_tweet_same_time, plot_ios_vs_studied, plot_comparison
import numpy as np
import sys
import os
from collections import defaultdict

In [ ]:
# Possible datasets
datasets=('Armenia', 'Catalonia', 'Ghana_Nigeria', 'Iran_5', 'Russia_3', 'Spain', 'Venezuela_2', 'Thailand', 'Ecuador', 'Iran_1', 'Iran_6', 'Russia_5', 'Qatar', 'Russia_2', 'China_1', 'China_2', 'Russia_1', 'Russia_4', 'Iran_2', 'Iran_3', 'Iran_4')

# Path to datasets directory (modify this if your data is stored elsewhere)
data_dir = "../data"

In [ ]:
#dataset = 'Catalonia'
for dataset in datasets:
    approach = "ignoring_tweet"  # Options: ignoring_tweet, shared_tweets, coretweets, coretweets_count

    print(f"\nDataset {dataset}")

    processed_dir = f"{data_dir}/{dataset}/Processed/"
    if not os.path.exists(processed_dir):
        print(f"Processed directory {processed_dir} does not exist.")
        sys.exit(1)

    RTs, io_users = import_data(processed_dir)
    print('Data imported.')

    pairs_coretweets = count_coretweets(RTs)
    users_coretweeted = set().union(*pairs_coretweets)

    RTs = filter_RTs(RTs, users_coretweeted)

    if approach == "ignoring_tweet":
        pairs_scores = coincide_ignoring_tweet(RTs)
    elif approach == "shared_tweets":
        pairs_scores = coincide_shared_tweets(RTs)
    elif approach == "coretweets":
        pairs_scores = coincide_same_tweet_same_time(RTs)
    elif approach == "coretweets_count":
        pairs_scores = pairs_coretweets
    else:
        raise ValueError("Unknown approach")

    #plot_ios_vs_studied(pairs_scores, io_users)
    plot_comparison(pairs_scores, pairs_coretweets, io_users)

## Only consider pairs with min_coretweets>=2

In [ ]:
#dataset = 'Catalonia'
for dataset in datasets:
    approach = "ignoring_tweet"  # Options: ignoring_tweet, shared_tweets, coretweets, coretweets_count

    print(f"\nDataset {dataset}")

    processed_dir = f"{data_dir}/{dataset}/Processed/"
    if not os.path.exists(processed_dir):
        print(f"Processed directory {processed_dir} does not exist.")
        sys.exit(1)

    RTs, io_users = import_data(processed_dir)
    print('Data imported.')

    pairs_coretweets = count_coretweets(RTs, min_coactions=2)
    users_coretweeted = set().union(*pairs_coretweets)

    RTs = filter_RTs(RTs, users_coretweeted)

    if approach == "ignoring_tweet":
        pairs_scores = coincide_ignoring_tweet(RTs)
    elif approach == "shared_tweets":
        pairs_scores = coincide_shared_tweets(RTs)
    elif approach == "coretweets":
        pairs_scores = coincide_same_tweet_same_time(RTs)
    elif approach == "coretweets_count":
        pairs_scores = pairs_coretweets
    else:
        raise ValueError("Unknown approach")

    #plot_ios_vs_studied(pairs_scores, io_users)
    plot_comparison(pairs_scores, pairs_coretweets, io_users)

### For pairs with same len(clusters), sort by # coretweets

In [ ]:
#dataset = 'Catalonia'
for dataset in datasets:
    approach = "ignoring_tweet"  # Options: ignoring_tweet, shared_tweets, coretweets, coretweets_count

    print(f"\nDataset {dataset}")

    processed_dir = f"{data_dir}/{dataset}/Processed/"
    if not os.path.exists(processed_dir):
        print(f"Processed directory {processed_dir} does not exist.")
        sys.exit(1)

    RTs, io_users = import_data(processed_dir)
    print('Data imported.')

    pairs_coretweets = count_coretweets(RTs)
    users_coretweeted = set().union(*pairs_coretweets)

    RTs = filter_RTs(RTs, users_coretweeted)

    if approach == "ignoring_tweet":
        pairs_scores = coincide_ignoring_tweet(RTs)
    elif approach == "shared_tweets":
        pairs_scores = coincide_shared_tweets(RTs)
    elif approach == "coretweets":
        pairs_scores = coincide_same_tweet_same_time(RTs)
    elif approach == "coretweets_count":
        pairs_scores = pairs_coretweets
    else:
        raise ValueError("Unknown approach")
    
    max_coretweets = max(pairs_coretweets.values()) + 1
    pairs_coretweets_extended = {}

    for pair in pairs_scores:
        if pair not in pairs_coretweets:
            pair_coretweet = 0
        else:
            pair_coretweet = pairs_coretweets[pair]
        pairs_coretweets_extended[pair] = pair_coretweet / max_coretweets

    #pairs_coretweets_extended = {p: pairs_coretweets.get(p, 0) for p in pairs_scores}
    #pairs_coretweets_extended = {p: v/(max(pairs_coretweets_extended.values())+1) for p, v in pairs_coretweets_extended.items()}
    pairs_scores = {p: v + pairs_coretweets_extended[p] for p, v in pairs_scores.items()}

    #plot_ios_vs_studied(pairs_scores, io_users)
    plot_comparison(pairs_scores, pairs_coretweets, io_users)

## Lets train a neural network with the three BASIC approaches

### Loss function should be improved, since maximizing the score of the IO pairs is not the best approach (we should maximize the maximum score of the io users)

In [ ]:
import os
import itertools
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from typing import Dict, List, Tuple, Optional
from synchronous_repeated_detection import _compute_curve

# Goal:
# Learn parameters a and b for score = a*pairs_ignoring + b*pairs_shared + 1*pairs_coretweet.
# Training: use 3 datasets as train, the rest as test; rotate the 3-dataset train set (cross-validation).
# Loss: L = mean(score(non_io)) - mean(score(io)) - 0.5 * mean(score(half_io))
# Evaluation: sort pairs by score, greedily collect unique users from pairs in order; count IO found vs users studied.
#            Compute area difference between ideal curve and obtained curve as the metric (lower is better).

# Safety: if the global variable `datasets` isn't present, define a default tuple
try:
    datasets
except NameError:
    datasets = (
        'Armenia', 'Catalonia', 'Ghana_Nigeria', 'Iran_5', 'Russia_3', 'Spain', 'Venezuela_2',
        'Thailand', 'Ecuador', 'Iran_1', 'Iran_6', 'Russia_5', 'Qatar', 'Russia_2',
        'China_1', 'China_2', 'Russia_1', 'Russia_4', 'Iran_2', 'Iran_3', 'Iran_4'
    )

# Use data_dir from previous cell, or default
try:
    data_dir
except NameError:
    data_dir = "../data"

# -----------------------------
# Data preparation helpers
# -----------------------------

def load_dataset_features(dataset: str):
    """
    Load a labeled dataset and compute pair features and labels.
    Returns dict with:
      - X: np.ndarray of shape (N, 3) with features [ignoring, shared, core]
      - y: np.ndarray of shape (N,) with labels in {0.0, 0.5, 1.0}
      - pairs: List[Tuple[user, user]] aligned with rows in X/y
      - io_users: Set of IO users for this dataset
    If dataset path doesn't exist or no pairs are produced, returns None.
    """
    processed_dir = f"{data_dir}/{dataset}/Processed/"
    if not os.path.exists(processed_dir):
        print(f"[WARN] Skipping missing dataset directory: {processed_dir}")
        return None

    RTs, io_users = import_data(processed_dir)

    pairs_coretweets = count_coretweets(RTs)
    if pairs_coretweets:
        users_coretweeted = set().union(*pairs_coretweets)
    else:
        users_coretweeted = set()

    RTs = filter_RTs(RTs, users_coretweeted)

    pairs_ignoring = coincide_ignoring_tweet(RTs)
    pairs_shared = coincide_shared_tweets(RTs)
    pairs_coretweet = coincide_same_tweet_same_time(RTs)

    common_pairs = set(pairs_ignoring) & set(pairs_shared) & set(pairs_coretweet)
    if not common_pairs:
        print(f"[WARN] No common pairs for dataset {dataset}")
        return None

    X_list = []
    y_list = []
    pairs_list = []

    for p in common_pairs:
        ign = pairs_ignoring[p]
        sha = pairs_shared[p]
        cor = pairs_coretweet[p]
        X_list.append([ign, sha, cor])
        u, v = p
        u_io = (u in io_users)
        v_io = (v in io_users)
        if u_io and v_io:
            lbl = 1.0
        elif u_io ^ v_io:
            lbl = 0.5
        else:
            lbl = 0.0
        y_list.append(lbl)
        pairs_list.append(p)

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)

    return {"X": X, "y": y, "pairs": pairs_list, "io_users": set(io_users)}


def build_train_tensor(train_names: List[str], cache: Dict[str, dict]):
    Xs = []
    ys = []
    for name in train_names:
        item = cache.get(name)
        if item is None:
            continue
        Xs.append(item["X"])
        ys.append(item["y"]) 
    if not Xs:
        return None, None
    X = np.vstack(Xs)
    y = np.concatenate(ys)
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)  # shape [N]
    return X, y


# -----------------------------
# Model and loss
# -----------------------------
class TwoParamLinear(nn.Module):
    """score = a*ignoring + b*shared + 1*core (no bias)."""
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(2))  # [a, b]
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # X shape: [N, 3] -> return [N]
        return X[:, :2] @ self.w + X[:, 2]


def requested_setwise_loss(scores: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Implements L = mean(score(non_io)) - mean(score(io)) - 0.5*mean(score(half_io)).
    If a group is empty, its mean is treated as 0 (i.e., contributes nothing).
    """
    masks = {
        1.0: (y == 1.0),
        0.5: (y == 0.5),
        0.0: (y == 0.0)
    }
    means: Dict[float, torch.Tensor] = {}

    loss = torch.tensor(0.0, dtype=scores.dtype, device=scores.device)

    # non-io
    if masks[0.0].any():
        loss = loss + scores[masks[0.0]].mean()
    # io
    if masks[1.0].any():
        loss = loss - scores[masks[1.0]].mean()
    # half
    if masks[0.5].any():
        loss = loss - 0.5 * scores[masks[0.5]].mean()

    return loss


# -----------------------------
# Evaluation: grouped user-discovery curve and area difference
# -----------------------------


def grouped_area_diff_to_ideal(pairs: List[Tuple], scores: np.ndarray, io_users: set):
    """
    Convert (pairs, scores) to pairs_scores dict, compute grouped curve, and
    return (area_diff, norm_area_diff, curves_dict).
    """
    if len(pairs) == 0:
        return np.nan, np.nan, {"x": [], "y": [], "ideal": []}
    pairs_scores = {p: float(scores[i]) for i, p in enumerate(pairs)}

    x_vals, y_vals = _compute_curve(pairs_scores, io_users)

    S = int(x_vals[-1]) if len(x_vals) > 0 else 0
    users_in_pairs = set([u for p in pairs for u in p])
    I = len(users_in_pairs & io_users)

    # Ideal curve: min(x, I) from x=0..S (align with x_vals length)
    ideal = [min(x, I) for x in range(0, S + 1)]

    # Compute area as sum over discrete steps (difference wrt ideal baseline 0)
    area_obtained = float(np.sum(y_vals))
    area_ideal = float(np.sum(ideal))

    area_diff = area_ideal - area_obtained
    norm_area_diff = (area_diff / area_ideal) if area_ideal > 0 else np.nan

    return area_diff, norm_area_diff, {"x": x_vals.tolist(), "y": y_vals.tolist(), "ideal": ideal}


# -----------------------------
# Cross-validation training and evaluation
# -----------------------------

# Preload all datasets once
cache: Dict[str, Optional[dict]] = {}
for name in datasets:
    cache[name] = load_dataset_features(name)

# Remove datasets that failed to load
valid_datasets = [d for d in datasets if cache.get(d) is not None]
print(f"Loaded {len(valid_datasets)} datasets out of {len(datasets)}.")

if len(valid_datasets) < 4:
    print("[ERROR] Need at least 4 valid datasets for 3-train vs rest-test cross-validation.")
else:
    # All combinations of 3 training datasets. Warning: can be large. Optionally cap the number for speed.
    all_combos = list(itertools.combinations(valid_datasets, 3))

    # Optional cap for runtime. Set to None to run all.
    max_folds = None  # e.g., 10 to limit; None for all
    if max_folds is not None:
        random.seed(0)
        random.shuffle(all_combos)
        all_combos = all_combos[:max_folds]

    fold_summaries = []

    for fold_idx, train_names in enumerate(all_combos, 1):
        test_names = [d for d in valid_datasets if d not in train_names]
        print(f"\n=== Fold {fold_idx}/{len(all_combos)} ===")
        print(f"Train on: {train_names}")
        print(f"Test on: {len(test_names)} datasets")

        X_train, y_train = build_train_tensor(list(train_names), cache)
        if X_train is None or len(y_train) == 0:
            print("[WARN] Empty training set; skipping fold.")
            continue

        # Model and optimizer
        model = TwoParamLinear()
        optimizer = optim.Adam(model.parameters(), lr=0.05)

        # Full-batch training (2 params, cheap)
        epochs = 1000
        for epoch in range(epochs):
            scores = model(X_train)
            loss = requested_setwise_loss(scores, y_train)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # Optionally clip extreme values for stability
            with torch.no_grad():
                if torch.any(torch.isnan(model.w)):
                    model.w[:] = torch.zeros_like(model.w)
            if (epoch + 1) % 200 == 0:
                a_val, b_val = model.w.detach().numpy().tolist()
                print(f"Epoch {epoch+1}/{epochs} | Loss={loss.item():.6f} | a={a_val:.4f}, b={b_val:.4f}")

        a_val, b_val = model.w.detach().numpy().tolist()
        print(f"Learned weights (fold {fold_idx}): a={a_val:.6f}, b={b_val:.6f} (core fixed to 1.0)")

        # Evaluate on test datasets using grouped curve area vs ideal
        fold_metrics = []
        for test_name in test_names:
            item = cache[test_name]
            X_test = torch.tensor(item["X"], dtype=torch.float32)
            with torch.no_grad():
                scores_t = model(X_test).cpu().numpy()
            area_diff, norm_area_diff, curves = grouped_area_diff_to_ideal(item["pairs"], scores_t, item["io_users"]) 
            fold_metrics.append({
                "dataset": test_name,
                "area_diff": area_diff,
                "norm_area_diff": norm_area_diff,
                "users": len({u for p in item["pairs"] for u in p}),
                "pairs": len(item["pairs"]) 
            })

        # Summarize fold
        valid_ms = [m for m in fold_metrics if not np.isnan(m["norm_area_diff"]) ]
        avg_norm = float(np.mean([m["norm_area_diff"] for m in valid_ms])) if valid_ms else np.nan
        print(f"Fold {fold_idx} avg normalized area diff over tests: {avg_norm:.6f}")

        fold_summaries.append({
            "train": train_names,
            "a": a_val,
            "b": b_val,
            "avg_norm_area_diff": avg_norm
        })

    # Final summary
    if fold_summaries:
        all_avgs = [fs["avg_norm_area_diff"] for fs in fold_summaries if not np.isnan(fs["avg_norm_area_diff"])]
        overall_avg = float(np.mean(all_avgs)) if all_avgs else np.nan
        print("\n==== Cross-validation summary ====")
        print(f"Folds run: {len(fold_summaries)}")
        best_fold = None
        best_val = float('inf')
        for fs in fold_summaries:
            a_val, b_val = fs["a"], fs["b"]
            avg_norm = fs["avg_norm_area_diff"]
            print(f"Train {fs['train']} -> avg norm area diff = {avg_norm:.6f} | a={a_val:.6f}, b={b_val:.6f}")
            if not np.isnan(avg_norm) and avg_norm < best_val:
                best_val = avg_norm
                best_fold = fs
        print(f"Overall mean of fold averages: {overall_avg:.6f}")
        if best_fold is not None:
            print(f"Best fold by avg norm area diff: train={best_fold['train']} | a={best_fold['a']:.6f}, b={best_fold['b']:.6f} | avg={best_fold['avg_norm_area_diff']:.6f}")
    else:
        print("No folds produced results.")